# NHANES Hv2 Colab Experiment\n\n这个 notebook 只负责实验代码执行，不伪造结果，不生成论文第4章正文。  \nThis notebook only runs experiment code. It does not fabricate results and does not generate thesis Chapter 4 text.\n\n固定输入路径 / Fixed input path: `/content/drive/MyDrive/nhanes_2021_2023_health_index_experiment/data`  \n固定输出路径 / Fixed output path: `/content/drive/MyDrive/nhanes_2021_2023_health_index_experiment/outputs`\n

In [ ]:
# 1. 挂载 Google Drive / Mount Google Drive\nfrom google.colab import drive\ndrive.mount('/content/drive')\n

In [ ]:
# 2. 安装依赖 / Install packages\nREPO_ROOT = '/content/nhanes_2021_2023_health_index_experiment'\n!if [ ! -f "$REPO_ROOT/requirements_colab.txt" ]; then echo "Repository not found at $REPO_ROOT. Clone the repository there before continuing."; else pip install -q -r "$REPO_ROOT/requirements_colab.txt"; fi\n

In [ ]:
# 3. 导入库 / Import libraries\nimport json\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nimport pandas as pd\n

In [ ]:
# 4. 定义路径 / Define paths\nPROJECT_ROOT = Path(REPO_ROOT)\nDATA_DIR = Path('/content/drive/MyDrive/nhanes_2021_2023_health_index_experiment/data')\nOUTPUT_DIR = Path('/content/drive/MyDrive/nhanes_2021_2023_health_index_experiment/outputs')\n\nassert PROJECT_ROOT.exists(), f'Repository path does not exist: {PROJECT_ROOT}'\nassert DATA_DIR.exists(), f'Data path does not exist: {DATA_DIR}'\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n\nrequired_files = [\n    DATA_DIR / 'adult_full_feature_set_v2.csv',\n    DATA_DIR / 'adult_reduced_feature_set_v2.csv',\n    DATA_DIR / 'adult_targets_v2.csv',\n]\nfor required_file in required_files:\n    assert required_file.exists(), f'Missing required file: {required_file}'\n\nprint(json.dumps({\n    'project_root': str(PROJECT_ROOT),\n    'data_dir': str(DATA_DIR),\n    'output_dir': str(OUTPUT_DIR),\n    'required_files': [str(path) for path in required_files],\n}, indent=2, ensure_ascii=False))\n

In [ ]:
# 5. 运行数据完整性与泄漏检查 / Run data integrity and leakage checks\nsubprocess.run([\n    sys.executable,\n    str(PROJECT_ROOT / 'scripts' / '01_data_check.py'),\n    '--data-dir', str(DATA_DIR),\n    '--output-dir', str(OUTPUT_DIR),\n], check=True)\n\ndata_check_path = OUTPUT_DIR / 'data_check' / 'data_check_report.md'\nprint(data_check_path.read_text(encoding='utf-8'))\n

## H_v2 Regression Runs\n\n下面的单元格用于训练 `H_v2` 回归模型。  \nThe cells below train `H_v2` regression models.\n

In [ ]:
# 训练 full 与 reduced 特征集 / Train full and reduced feature sets\nfor feature_set in ('full', 'reduced'):\n    subprocess.run([\n        sys.executable,\n        str(PROJECT_ROOT / 'scripts' / '02_train_hv2_regression.py'),\n        '--feature-set', feature_set,\n        '--data-dir', str(DATA_DIR),\n        '--output-dir', str(OUTPUT_DIR),\n    ], check=True)\n

In [ ]:
# 年龄组稳定性与 H_v1 敏感性分析 / Run age-group stability and H_v1 sensitivity\nfor feature_set in ('full', 'reduced'):\n    subprocess.run([\n        sys.executable,\n        str(PROJECT_ROOT / 'scripts' / '03_age_group_stability.py'),\n        '--feature-set', feature_set,\n        '--model', 'random_forest',\n        '--data-dir', str(DATA_DIR),\n        '--output-dir', str(OUTPUT_DIR),\n    ], check=True)\n    subprocess.run([\n        sys.executable,\n        str(PROJECT_ROOT / 'scripts' / '04_hv1_sensitivity.py'),\n        '--feature-set', feature_set,\n        '--model', 'random_forest',\n        '--data-dir', str(DATA_DIR),\n        '--output-dir', str(OUTPUT_DIR),\n    ], check=True)\n

In [ ]:
# SHAP 分析与实验总结 / Run SHAP analysis and experiment summary\nsubprocess.run([\n    sys.executable,\n    str(PROJECT_ROOT / 'scripts' / '05_shap_analysis.py'),\n    '--feature-set', 'full',\n    '--data-dir', str(DATA_DIR),\n    '--output-dir', str(OUTPUT_DIR),\n], check=True)\n\nsubprocess.run([\n    sys.executable,\n    str(PROJECT_ROOT / 'scripts' / '06_generate_experiment_summary.py'),\n    '--output-dir', str(OUTPUT_DIR),\n], check=True)\n\nsummary_path = OUTPUT_DIR / 'experiment_summary' / 'experiment_summary.md'\nprint(summary_path.read_text(encoding='utf-8'))\n